In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Section 3.2 — Simple Regression Models

**Objectives**

By the end of this section you will be able to:

- Train a Linear Regression model to predict a continuous numerical outcome.
- Evaluate regression performance using MAE, RMSE, and R².
- Interpret regression coefficients in a business context.
- Visualise actual versus predicted values to assess model fit.

> **Cooking analogy:** Regression is like **reverse-engineering a recipe to hit
> a target flavour score**. You adjust ingredient quantities (feature values) and
> the model learns which changes move the final score (the target) up or down —
> and by exactly how much. Once trained, hand it a new set of quantities and it
> predicts the score before you even light the stove.

**Regression** predicts a **continuous** numerical outcome (e.g., revenue,
house price, demand).

### Linear Regression

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_n x_n$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(5)
n = 400

# House price dataset
df_house = pd.DataFrame({
    "Size_sqft"  : np.random.randint(500, 4000, n),
    "Bedrooms"   : np.random.randint(1, 6, n),
    "Age_years"  : np.random.randint(0, 50, n),
    "Distance_km": np.random.uniform(1, 30, n).round(1),
})

# Price = true relationship + noise
df_house["Price"] = (
      250 * df_house["Size_sqft"]
    + 15_000 * df_house["Bedrooms"]
    - 2_000 * df_house["Age_years"]
    - 5_000 * df_house["Distance_km"]
    + 80_000
    + np.random.normal(0, 30_000, n)
).round(-2)

print("House dataset:")
print(df_house.describe().round(0))

In [ ]:
X = df_house.drop(columns="Price")
y = df_house["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Train the model
model_lr = LinearRegression()
model_lr.fit(X_train_sc, y_train)

# Predict
y_pred = model_lr.predict(X_test_sc)

In [ ]:
# Evaluate performance
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"Mean Absolute Error  : ${mae:,.0f}")
print(f"Root Mean Sq. Error  : ${rmse:,.0f}")
print(f"R² Score             : {r2:.3f}")

In [ ]:
# Visualise actual vs predicted
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, y_pred, alpha=0.4, color="steelblue")
ax.plot([y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()], "r--", label="Perfect prediction")
ax.set_xlabel("Actual Price ($)")
ax.set_ylabel("Predicted Price ($)")
ax.set_title("Linear Regression — Actual vs Predicted")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Regression coefficients — feature impact
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model_lr.coef_
}).sort_values("Coefficient", ascending=False)

print("Feature Coefficients (standardised):")
print(coef_df.to_string(index=False))
print("\nInterpretation: larger absolute value = stronger influence on price.")

### Interpreting Regression Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **MAE** | mean(|actual − predicted|) | Average dollar error |
| **RMSE** | √mean((actual−predicted)²) | Penalises large errors more |
| **R²** | 1 − SS_res/SS_tot | 0 = no fit; 1 = perfect fit |

---

### Student Task 3.2

A marketing team wants to predict next month's **advertising spend** required
to achieve a target **sales volume**.

1. Create a synthetic dataset (100 rows) with features `Ad_Budget`, `Season` (encode as 1–4), and `Competitors` (integer), and a target `Sales`.
2. Train a `LinearRegression` model and evaluate with MAE, RMSE, and R².
3. Plot Actual vs Predicted sales.
4. Which feature has the largest coefficient? What does that mean for the business?

In [ ]:
# Your code here

---

### Evaluation Questions 3.2

1. R² = 0.85 means the model explains what percentage of variance in the target?
   a) 15 %
   b) 85 % (correct)
   c) 8.5 %
   d) 0.85 %

2. Which metric is most sensitive to large prediction errors?
   a) MAE
   b) R²
   c) RMSE (correct)
   d) Accuracy

3. Linear regression assumes the relationship between features and target is:
   a) Exponential
   b) Linear (correct)
   c) Circular
   d) Random

4. `model.coef_` returns:
   a) The model's accuracy score
   b) The number of training iterations
   c) The learned weights for each feature (correct)
   d) The predicted values

5. If a regression coefficient for `Ad_Budget` is positive, it means:
   a) Higher ad spend predicts lower sales
   b) Higher ad spend predicts higher sales (correct)
   c) Ad budget is unrelated to sales
   d) Ad budget should be removed from the model

---